In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import binom

def estimate_XY_em(counts, N, n_init=10, max_iter=500, tol=1e-8, seed=0):
    counts = np.asarray(counts, dtype=float)
    M = len(counts)
    rng = np.random.default_rng(seed)
    best = None
    for _ in range(n_init):
        Y = rng.uniform(0.01, 0.3)
        X = rng.uniform(0.5, 0.99)
        pi = rng.uniform(0.1, 0.5)
        prev_ll = -np.inf
        for _ in range(max_iter):
            log_pX = binom.logpmf(counts, N, np.clip(X, 1e-9, 1-1e-9))
            log_pY = binom.logpmf(counts, N, np.clip(Y, 1e-9, 1-1e-9))
            log_a = np.log(pi + 1e-300) + log_pX
            log_b = np.log(1 - pi + 1e-300) + log_pY
            m = np.maximum(log_a, log_b)
            denom = m + np.log(np.exp(log_a - m) + np.exp(log_b - m))
            resp = np.exp(log_a - denom)
            ll = np.sum(denom)
            s1 = resp.sum()
            s0 = (1 - resp).sum()
            X = (resp * counts).sum() / (N * s1 + 1e-12)
            Y = ((1 - resp) * counts).sum() / (N * s0 + 1e-12)
            pi = s1 / M
            X = np.clip(X, 1e-6, 1 - 1e-6)
            Y = np.clip(Y, 1e-6, 1 - 1e-6)
            pi = np.clip(pi, 1e-6, 1 - 1e-6)

            if abs(ll - prev_ll) < tol:
                break
            prev_ll = ll
        if X < Y:
            X, Y = Y, X
            pi = 1 - pi
            resp = 1 - resp
        if best is None or ll > best['loglik']:
            best = dict(X=X, Y=Y, pi=pi, resp=resp, loglik=ll)
    return best

df = pd.read_csv(r'RHO-Boruta.csv')
counts = (df.iloc[:, :] <= 2).sum(axis=0).values
M = len(df.columns)
res = estimate_XY_em(counts, 20, n_init=100, max_iter=1000)
print(f"估计 X = {res['X']:.4f}")
print(f"估计 Y = {res['Y']:.4f}")
print(f"估计重要特征比例 pi = {res['pi']:.4f}")

df_ = pd.DataFrame({
    'feature': df.columns,
    'count': counts.astype(int),
    'prob_important': res['resp']
})
# df_.sort_values(by='prob_important', ascending=False, inplace=True)
df_[df_['prob_important'] >= 0.95]

估计 X = 0.9855
估计 Y = 0.0002
估计重要特征比例 pi = 0.0217


,feature,count,prob_important
31,RDKit2D_Chi0v,20,1.0
34,RDKit2D_Chi1v,20,1.0
121,RDKit2D_MolMR,20,1.0
740,Mordred2D_ATS0v,20,1.0
741,Mordred2D_ATS1v,20,1.0
742,Mordred2D_ATS2v,20,1.0
759,Mordred2D_ATS1pe,20,1.0
776,Mordred2D_ATS0p,20,1.0
777,Mordred2D_ATS1p,20,1.0
885,Mordred2D_ATSC0i,20,1.0
